In [ ]:
# ==============================================================================
# 📚 AI 코딩 튜터의 오늘의 실습 데이터셋: Moo/korean-parallel-corpora
#
# [데이터셋 설명]
# 이 데이터셋은 한국어(ko)와 영어(en)의 '병렬 코퍼스(Parallel Corpora)'입니다.
# 즉, 같은 내용의 문장 쌍이 여러 개 모여 있어요!
# 예: (한국어 문장, 영어 번역문장)
#
# [💡 실습 목표]
# 우리는 이 데이터를 활용해서 '번역의 패턴'을 분석하고, 마치 AI가 작동하는 것처럼
# 입력된 한국어 문장으로 최고의 영어 프롬프트를 만드는 과정을 시뮬레이션 해볼 거예요.
#
# ✨ 컨셉: "AI 통역사 챌린지 - 가장 자연스러운 프롬프트 찾기!"
# ==============================================================================

import random
import time
from datasets import load_dataset, get_dataset_config_names
from tqdm import tqdm

# 🚀 우리가 사용할 데이터셋 ID와 기본 설정
DATASET_NAME = "Moo/korean-parallel-corpora"
# 전체 데이터를 한 번에 로드하는 것은 비효율적이므로, 학습용 트레인셋을 스트리밍 방식으로 로드합니다.
DATASET_SPLIT = 'train'
SAMPLE_COUNT = 100  # 전체 데이터셋에서 무작위로 뽑아 실습할 샘플 개수
MAX_SAMPLES = 1000 # 로딩 실패 시 백업으로 사용할 최대 샘플 개수

print("===============================================================")
print("✨ 튜터가 알려줄게! 병렬 코퍼스 실습을 시작할게! 🤩")
print("===============================================================")

# 1. 데이터셋 로딩 준비
# 데이터셋 이름만 먼저 받아와서 사용 가능한 설정을 확인해봅니다.
try:
    # 데이터셋 ID를 사용하여 설정을 확인합니다.
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    selected_config = None # 이 데이터셋은 config가 복잡하지 않아 빈 값으로 둡니다.
except Exception as e:
    print(f"ℹ️ 데이터셋 {DATASET_NAME}은 별도의 Config가 없거나 기본(default) 설정만 제공됩니다. 진행할게.")
    selected_config = None

dataset = None
try:
    # 2. 스트리밍 모드로 데이터 로드 시도 (빠른 진행을 위한 필수 기법!)
    print(f"\n>>> 📡 스트리밍 모드 ({DATASET_NAME}, split='{DATASET_SPLIT}')로 로드 시도 중...")
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    print("🎉 성공! 스트리밍 모드로 데이터를 불러왔어! 아주 빠르지?")

except Exception as e:
    # 스트리밍 모드가 불안정할 경우, 일반 모드로 소량만 다운로드하여 진행합니다.
    print(f"\n⚠️ 경고: 스트리밍 모드 로드에 실패했어. ({e})")
    print(f"   -> 대신, 'test' 스플릿에서 {MAX_SAMPLES}개만 일반 모드로 다운로드해서 진행할게. 😊")
    try:
        dataset = load_dataset(DATASET_NAME, split='test', streaming=False)
    except Exception as e2:
        print(f"❌ 최후의 수단까지 실패했어. 데이터셋 로딩을 중단할게. {e2}")
        exit()

# 3. 실제 학습에 사용할 샘플 추출
print("\n===============================================================")
print(f"✨ 이제 무작위로 {SAMPLE_COUNT}개의 샘플을 뽑아서 실습할 거야!")
print("===============================================================")

# 스트리밍 데이터셋인지 확인하고 적절한 샘플링 로직을 적용합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)일 가능성이 높습니다.
    print("💡 스트리밍 모드 감지: dataset.take() 사용!")
    # 스트리밍 데이터를 순회하며 필요한 만큼만 가져옵니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
    # Iterator를 리스트로 변환하여 반복 가능하도록 합니다.
    sampled_dataset = list(sample_iterator)
else:
    # 일반 Dataset인 경우
    print("💡 일반 Dataset 감지: dataset.take() 사용!")
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sampled_dataset = list(sample_iterator)


# 4. 🌟 창의적 실습 시작: 번역 스타일 패턴 분석기 (AI 프롬프트 생성 시뮬레이션)
print("\n###############################################################")
print("# 🔍 [미니 프로젝트] AI 통역사 스타일 패턴 분석기 가동! 🤖")
print("###############################################################")

def analyze_sample(sample):
    """하나의 샘플 쌍을 받아 패턴을 분석하고 흥미로운 정보를 출력합니다."""
    ko_text = sample['ko']
    en_text = sample['en']

    print("-" * 50)
    print(f"💡 원문 (KO): {ko_text[:40]}...")
    print(f"💡 번역 (EN): {en_text[:40]}...")

    # A. 길이에 대한 흥미로운 비교 (정량적 분석)
    ko_len = len(ko_text.split())
    en_len = len(en_text.split())
    print(f"    [📊 분석] KO 단어 수: {ko_len}개 | EN 단어 수: {en_len}개")

    # B. 번역 스타일 분석 (가장 재미있는 부분!)
    if ko_len > 5 and en_len > 5 and (ko_len > en_len or en_len > ko_len):
        # 단어 수 차이가 크면, '문체 변화'가 일어난 경우입니다.
        print("    ⭐ [Insight] 단어 수 차이가 커요! (문체가 변했거나, 간결하게 축약됐을 수 있어요!)")
    else:
        print("    ✨ [Insight] 단어 수가 비슷해요. (일대일 번역에 충실하네요!)")

    # C. LLM 프롬프트 생성 시뮬레이션 (가장 실용적인 결과!)
    # 만약 우리가 이 데이터를 학습한 AI라면, 이 패턴을 이용해서 '개선된 프롬프트'를 만들 수 있어요.
    print("\n    ✨ [AI 시뮬레이션] 이 데이터를 기반으로 생성할 만한 프롬프트:")
    print("    >>> Prompt: '한국어 문장을 다음의 스타일(Formal/Informal/Academic 등)로 번역해줘.'")
    print(f"    >>> 예시 (입력 KO): '{ko_text}'")
    print(f"    >>> 기대 결과 (출력 EN): '{en_text}'\n")


# 5. 샘플 반복 및 분석
print("\n\n=================== 🔬 실습 데이터셋 분석 시작! 🔬 ================")
if sampled_dataset:
    # tqdm을 사용하여 진행 바를 보여주면, 코드가 열심히 작동하는 느낌을 줄 수 있어요!
    for sample in tqdm(sampled_dataset, desc="✨ 패턴 분석 중"):
        try:
            analyze_sample(sample)
        except KeyError as e:
            print(f"\n[경고] 샘플에서 필요한 키 '{e}'를 찾을 수 없습니다. 다음 샘플로 넘어가요.")
            continue
else:
    print("\n🚨 분석할 샘플이 준비되지 않았어요. 데이터 로딩을 확인해주세요!")

print("\n===============================================================")
print("🎉 축하해! AI 코딩 실습을 완벽하게 마쳤어! 👍")
print("데이터의 구조와 패턴을 분석하는 능력이 정말 멋지다!")
print("===============================================================")